# 03 Contrast Logic Demo

This notebook demonstrates how the MA-related contrast logic from the original MATLAB fNIRS pipeline is translated into Python.

This notebook does not run the full GLM pipeline. Instead, it focuses on verifying the contrast matrix used for MA versus Control comparisons.


## MATLAB contrast logic

The original MATLAB group-level model uses six condition columns:

1. G4_6 Control
2. G1_3 Control
3. G4_6 MA
4. G1_3 MA
5. G4_6 PA
6. G1_3 PA

The main MA-related contrasts are:

1. G4_6 MA - G4_6 Control
2. G1_3 MA - G1_3 Control
3. (G4_6 MA - G4_6 Control) - (G1_3 MA - G1_3 Control)


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Make sure Python can find modules in the src/ folder
cwd = Path.cwd()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(project_root / 'src'))

from contrasts import CONDITION_ORDER, CONTRASTS, get_contrast


## Step 1: Check condition order

The condition order must match the MATLAB group-level model output. If the order is wrong, the contrast results will also be wrong.


In [ ]:
condition_df = pd.DataFrame({
    'column_index': range(1, len(CONDITION_ORDER) + 1),
    'condition': CONDITION_ORDER
})

condition_df


## Step 2: Display contrast matrix

Each contrast vector contains one weight for each condition column.

For example, `[-1, 0, 1, 0, 0, 0]` means:

G4_6 MA - G4_6 Control.


In [ ]:
contrast_df = pd.DataFrame(
    {name: vector for name, vector in CONTRASTS.items()},
    index=CONDITION_ORDER
).T

contrast_df


## Step 3: Apply contrasts to example beta values

In the real pipeline, beta values will come from the group-level GLM output.

Here, we use example beta values only to demonstrate how contrast values are computed.


In [ ]:
example_betas = pd.Series(
    [0.10, 0.08, 0.35, 0.20, 0.22, 0.18],
    index=CONDITION_ORDER,
    name='example_beta'
)

example_betas


In [ ]:
contrast_results = []

for name, vector in CONTRASTS.items():
    contrast_value = np.dot(vector, example_betas.values)
    contrast_results.append({
        'contrast': name,
        'contrast_value': contrast_value
    })

contrast_results_df = pd.DataFrame(contrast_results)
contrast_results_df


## Interpretation

This notebook verifies that the Python contrast vectors reproduce the MA-related contrast logic from the MATLAB pipeline.

The next step is to connect these contrast definitions to actual group-level GLM outputs or a processed result table.
